# Problema 4 — Otimização mista da rede hidráulica

Este notebook apresenta a implementação do **Problema 4**, no qual existe um número limitado de canos grossos, de menor resistência hidráulica, e é necessário decidir em quais arestas da rede eles devem ser instalados.

A métrica otimizada é a maior pressão nodal da rede. Como a pressão de saída é prescrita, a menor pressão deixa de ser uma variável de projeto e passa a ser uma condição de contorno conhecida. Assim, o objetivo computacional é

\[
\min \; p^{\max}
\qquad \text{com} \qquad
p_i\leq p^{\max},\; i\in V.
\]

A apresentação contém três estudos computacionais:

1. **Caso 1:** rede com 20 nós, usada para mostrar a resolução por MILP;
2. **Caso 2:** rede com 100 nós e poucas arestas, mantendo o problema maior, mas tratável;
3. **Caso 3:** mesmo tipo de formulação com limite de tempo artificialmente muito baixo, mostrando que o código retorna a melhor solução viável encontrada ou uma solução heurística de fallback.


## 1. Formulação usada

Para cada aresta \(e\in E\), define-se a variável binária

\[
x_e=
\begin{cases}
1, & \text{se a aresta recebe cano grosso},\\
0, & \text{se a aresta permanece com cano fino}.
\end{cases}
\]

Se existem \(\mathcal{C}\) canos grossos disponíveis, impõe-se

\[
\sum_{e\in E}x_e=\mathcal{C}.
\]

A lei constitutiva depende da escolha do tipo de cano:

\[
q_e=k_e^f(Mp)_e, \quad \text{se } x_e=0,
\]

\[
q_e=k_e^g(Mp)_e, \quad \text{se } x_e=1.
\]

Essa condição lógica é implementada com restrições Big-\(M\). O objetivo é minimizar \(p^{\max}\), que representa a maior pressão nodal produzida pela configuração escolhida de canos grossos. A restrição \(p_i\leq p^{\max}\) não é um limite físico adicional; ela apenas define a variável auxiliar que aparece na função objetivo.


## 2. Importações

Para este notebook funcionar, o arquivo `ProblemaP4.py` deve estar dentro da pasta `src/` do projeto. A versão corrigida da classe já minimiza diretamente a pressão máxima da rede.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.ProblemGeneratorP1 import ProblemaP1Generator
from src.ProblemaP4 import ProblemaP4

## 4. Funções auxiliares da apresentação

As funções abaixo reduzem repetição no notebook. Elas geram uma rede, resolvem o P1 de referência, resolvem o P4 e exibem tabelas comparativas.

In [ ]:
def gerar_rede(seed, num_nodes, edge_prob):
    generator = ProblemaP1Generator(seed=seed)
    p1 = generator.generate(
        num_nodes=num_nodes,
        edge_prob=edge_prob,
        mu=1e-3,
        patm=0.0,
        q_mode="uniform",
        single_sink=True,
    )
    p1.setup()
    p1.solve()
    return p1


def mostrar_resumo_p1(p1, nome):
    print(nome)
    print("numero de nos:", p1.num_nodes)
    print("numero de arestas:", p1.num_edges)
    print("pressao minima p1:", float(np.min(p1.p)), "Pa")
    print("pressao maxima p1:", float(np.max(p1.p)), "Pa")
    print("amplitude de pressao p1:", float(np.max(p1.p) - np.min(p1.p)), "Pa")


def resolver_caso_p4(p1, nome, time_limit, mip_rel_gap=1e-4, disp=False):
    p4 = ProblemaP4(
        p1_instance=p1,
        max_thick_pipes=3,
        thick_area_factor=2.0,
        use_exact_number=True,
        pressure_bound_factor=5.0,
        big_m_factor=1.05,
        solver_options={
            "disp": disp,
            "time_limit": time_limit,
            "mip_rel_gap": mip_rel_gap,
        },
    )

    baseline = p4.evaluate_configuration(thick_edges=[])
    summary = p4.solve_milp(accept_feasible=True, use_fallback=True)

    comparison = {
        "caso": ["todos finos", "p4 otimizado"],
        "nos": [p1.num_nodes, p1.num_nodes],
        "arestas": [p1.num_edges, p1.num_edges],
        "pressao_minima_pa": [baseline["pressure_min"], summary["pressure_min"]],
        "pressao_maxima_pa": [baseline["pressure_max"], summary["objective_pressure_max"]],
        "amplitude_pressao_pa": [baseline["pressure_range"], summary["pressure_range"]],
    }

    reduction = (
        (baseline["pressure_max"] - summary["objective_pressure_max"])
        / baseline["pressure_max"]
        * 100
    )

    print(nome)
    print("status da solucao:", summary["solution_status"])
    print("solucao otima certificada:", summary["solution_is_optimal"])
    print("mensagem do solver:", summary["solver_message"])
    print("gap mip:", summary["mip_gap"])
    print("reducao percentual da pressao maxima:", reduction, "%")
    print("arestas grossas selecionadas:", summary["selected_thick_edges"])

    if pd is not None:
        comparison_df = pd.DataFrame(comparison)
        display(comparison_df)
    else:
        print(comparison)

    return p4, baseline, summary

## 5. Caso 1 — rede com 20 nós

Este caso é usado para apresentar a solução por otimização mista em uma rede de tamanho pequeno/médio. Ele mantém a ideia dos testes anteriores: poucos canos grossos são escolhidos entre todas as arestas disponíveis.

In [ ]:
p1_20 = gerar_rede(
    seed=42,
    num_nodes=20,
    edge_prob=0.10,
)

mostrar_resumo_p1(p1_20, "caso 1: rede com 20 nos")

In [ ]:
p4_20, baseline_20, summary_20 = resolver_caso_p4(
    p1=p1_20,
    nome="caso 1: p4 com 20 nos",
    time_limit=300,
    mip_rel_gap=1e-4,
    disp=False,
)

### 5.1. Arestas escolhidas no caso de 20 nós

In [ ]:
edge_rows_20 = p4_20.edge_solution_table()

if pd is not None:
    edge_df_20 = pd.DataFrame(edge_rows_20)
    display(edge_df_20.sort_values(["x_grosso", "edge"], ascending=[False, True]))
else:
    for row in edge_rows_20:
        print(row)

### 5.2. Pressões nodais no caso de 20 nós

In [ ]:
node_rows_20 = p4_20.node_solution_table()

if pd is not None:
    node_df_20 = pd.DataFrame(node_rows_20)
    display(node_df_20.sort_values("pressao", ascending=False))
else:
    for row in node_rows_20:
        print(row)

### 5.3. Visualização do caso de 20 nós

In [ ]:
p4_20.plot_solution(
    layout="spring",
    figsize=(11, 8),
    show_labels=True,
)
plt.show()

## 6. Caso 2 — rede com 100 nós e poucas arestas

Este caso mantém 100 nós, mas reduz a probabilidade de conexão para controlar o número de arestas. Isso é importante porque a dificuldade do problema misto cresce fortemente com o número de variáveis binárias, isto é, com o número de arestas.

In [ ]:
p1_100 = gerar_rede(
    seed=42,
    num_nodes=100,
    edge_prob=0.006,
)

mostrar_resumo_p1(p1_100, "caso 2: rede com 100 nos e poucas arestas")

In [ ]:
p4_100, baseline_100, summary_100 = resolver_caso_p4(
    p1=p1_100,
    nome="caso 2: p4 com 100 nos e poucas arestas",
    time_limit=300,
    mip_rel_gap=1e-4,
    disp=False,
)

### 6.1. Arestas escolhidas no caso de 100 nós

In [ ]:
edge_rows_100 = p4_100.edge_solution_table()

if pd is not None:
    edge_df_100 = pd.DataFrame(edge_rows_100)
    display(edge_df_100.sort_values(["x_grosso", "edge"], ascending=[False, True]))
else:
    for row in edge_rows_100:
        print(row)

### 6.2. Pressões nodais no caso de 100 nós

In [ ]:
node_rows_100 = p4_100.node_solution_table()

if pd is not None:
    node_df_100 = pd.DataFrame(node_rows_100)
    display(node_df_100.sort_values("pressao", ascending=False))
else:
    for row in node_rows_100:
        print(row)

## 7. Caso 3 — limite de tempo e retorno de solução não ótima

Este caso mostra a funcionalidade nova. O limite de tempo é colocado artificialmente muito baixo. Com isso, o solver pode parar antes de certificar otimalidade. A implementação então segue a ordem:

1. usa a solução ótima, se ela foi certificada;
2. aceita a melhor solução inteira viável retornada pelo solver, se existir;
3. se o solver não retornar solução viável, usa uma heurística baseada nas maiores quedas de pressão da rede com todos os canos finos.

O objetivo deste caso não é produzir o melhor resultado possível, mas mostrar que o notebook continua retornando uma configuração admissível.

In [ ]:
p4_timeout, baseline_timeout, summary_timeout = resolver_caso_p4(
    p1=p1_100,
    nome="caso 3: limite de tempo artificialmente baixo",
    time_limit=1e-9,
    mip_rel_gap=1e-4,
    disp=False,
)

### 7.1. Interpretação do status

A chave `solution_status` resume o tipo de solução retornada:

- `otima`: o solver certificou otimalidade global;
- `viavel_nao_certificada`: o solver parou antes da certificação, mas retornou uma solução inteira viável;
- `heuristica`: o solver não retornou solução inteira viável aproveitável e foi usada a heurística de fallback.

In [ ]:
summary_timeout

### 7.2. Arestas escolhidas no caso com limite de tempo

In [ ]:
edge_rows_timeout = p4_timeout.edge_solution_table()

if pd is not None:
    edge_df_timeout = pd.DataFrame(edge_rows_timeout)
    display(edge_df_timeout.sort_values(["x_grosso", "edge"], ascending=[False, True]))
else:
    for row in edge_rows_timeout:
        print(row)

## 8. Verificação por força bruta para redes pequenas

Para redes pequenas, é possível comparar a solução do MILP com uma busca exaustiva. Essa etapa é opcional, pois o número de combinações cresce rapidamente com o número de arestas.

In [ ]:
run_brute_force_check = False

if run_brute_force_check:
    brute = p4_20.brute_force_check(max_combinations=200000)
    print("pressao maxima p4:", p4_20.summary()["objective_pressure_max"])
    print("pressao maxima forca bruta:", brute["pressure_max"])
    print("arestas p4:", p4_20.summary()["selected_thick_edges"])
    print("arestas forca bruta:", brute["thick_edges"])